<a href="https://colab.research.google.com/github/huamanchristian44/LAB09/blob/develop/Laboratorio_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Laboratorio 09: Redes neuronales artificiales**

In [79]:
#Importando librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

In [80]:
#Cargando el dataset desde la URL
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
column_names = ['Sample_code_number', 'Clump_Thickness', 'Uniformity_of_Cell_Size',
                'Uniformity_of_Cell_Shape', 'Marginal_Adhesion', 'Single_Epithelial_Cell_Size',
                'Bare_Nuclei', 'Bland_Chromatin', 'Normal_Nucleoli', 'Mitoses', 'Class']
data = pd.read_csv(url, names=column_names)
data.head(10)

,Sample_code_number,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2
5,1017122,8,10,10,8,7,10,9,7,1,4
6,1018099,1,1,1,1,2,10,3,1,1,2
7,1018561,2,1,2,1,2,1,3,1,1,2
8,1033078,2,1,1,1,2,1,1,1,5,2
9,1033078,4,2,1,1,2,1,2,1,1,2


In [81]:
#Reemplazando '?' por NaN y convertir Bare_Nuclei a numérico
data.replace('?', np.nan, inplace=True)
data['Bare_Nuclei'] = pd.to_numeric(data['Bare_Nuclei'])

In [82]:
#Imputación de valores faltantes (mediana)
imputer = SimpleImputer(strategy="median")
data.iloc[:, 1:-1] = imputer.fit_transform(data.iloc[:, 1:-1])

In [83]:
#Separando las características (X) de la variable objetivo (y)
X = data.drop(['Sample_code_number', 'Class'], axis=1)
y = data['Class'].apply(lambda x: 1 if x == 4 else 0)

In [84]:
#Dividiendo los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [85]:
#Escalando las características para que tengan media 0 y desviación estándar 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [86]:
#Red neuronal inicial y ajuste de hiperparámetros
param_grid = {
    'hidden_layer_sizes': [(10,), (20,), (10,10), (20,10)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
    'max_iter': [500, 1000],
    'learning_rate': ['constant', 'adaptive']
}

mlp = MLPClassifier(random_state=42)

grid = GridSearchCV(mlp, param_grid, cv=5, scoring='accuracy', verbose=2, n_jobs=-1)
grid.fit(X_train, y_train)


Fitting 5 folds for each of 64 candidates, totalling 320 fits


GridSearchCV(cv=5, estimator=MLPClassifier(random_state=42), n_jobs=-1,
             param_grid={'activation': ['relu', 'tanh'],
                         'hidden_layer_sizes': [(10,), (20,), (10, 10),
                                                (20, 10)],
                         'learning_rate': ['constant', 'adaptive'],
                         'max_iter': [500, 1000], 'solver': ['adam', 'sgd']},
             scoring='accuracy', verbose=2)

In [67]:
#Determinando el mejor modelo
best_model = grid.best_estimator_
print("Mejores parámetros encontrados:", grid.best_params_)

Mejores parámetros encontrados: {'activation': 'tanh', 'hidden_layer_sizes': (10,), 'learning_rate': 'constant', 'max_iter': 1000, 'solver': 'adam'}


In [68]:
#Evaluación
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy en el conjunto de prueba:", round(accuracy, 4))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))


Accuracy en el conjunto de prueba: 0.9571

Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97        95
           1       0.95      0.91      0.93        45

    accuracy                           0.96       140
   macro avg       0.96      0.95      0.95       140
weighted avg       0.96      0.96      0.96       140

